# Agent Orchestration Patterns -- Banking Fraud Resolution (Azure AI Foundry Projects)

This notebook compares **5 orchestration patterns** against the same banking fraud scenario, but uses **persistent Azure AI Foundry agent definitions** created through `azure.ai.projects.AIProjectClient` and executes them through OpenAI chat completions instead of `azure.ai.agents.AgentsClient`.

> *A customer reports unauthorized transactions on their business checking account totaling $12,500 over the past 48 hours. They need fraud investigation, account security, and provisional credit.*

| Pattern | How It Works | Best For |
|---------|-------------|----------|
| **Sequential** | Agents run one after another, each building on previous output | Multi-step processes with dependencies |
| **Handoff** | Router analyzes request, delegates to a specialist agent | Dynamic routing to domain experts |
| **Concurrent** | Multiple agents analyze in parallel, results are merged | Independent analyses needed fast |
| **Group Chat** | Agents collaborate in rounds with shared context | Complex decisions requiring debate |
| **Magentic** | Planner decomposes into task ledger, researcher + writer + validator iterate | Complex cases needing structured decomposition and quality gates |


## 1. Setup -- Load environment and create `AIProjectClient`


In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
from azure.ai.inference.tracing import AIInferenceInstrumentor

load_dotenv(Path('.env'), override=True)

PROJECT_ENDPOINT = os.getenv('AZURE_AI_ENDPOINT')
MODEL_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')

if not PROJECT_ENDPOINT:
    raise ValueError('Set AZURE_AI_ENDPOINT (AI Foundry project endpoint) in .env')
if not MODEL_DEPLOYMENT:
    raise ValueError('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME is required in .env')

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# Enable tracing so runs appear in Foundry portal under Tracing
app_insights_conn = client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=app_insights_conn)
OpenAIInstrumentor().instrument()
AIInferenceInstrumentor().instrument()
print('Tracing enabled (Application Insights + OpenAI + Inference instrumentors).')

oai = client.get_openai_client()

BANKING_TASK = (
    'A customer reports unauthorized transactions on their business checking account '
    'totaling $12,500 over the past 48 hours. Three transactions were flagged: '
    '$4,200 wire transfer to an unknown account, $5,800 online purchase from an '
    'overseas merchant, and $2,500 ATM withdrawal in a different state. '
    'The customer needs immediate resolution including fraud investigation, '
    'account security measures, and provisional credit assessment.'
)

results = {}

print(f'Project endpoint: {PROJECT_ENDPOINT}')
print(f'Model deployment: {MODEL_DEPLOYMENT}')
print('AIProjectClient ready.')
print('OpenAI client ready.')


## 2. Helper -- Idempotent agent creation


In [14]:
def get_or_create_agent(client, name, instructions, model, metadata=None):
    metadata = metadata or {}

    for existing in client.agents.list():
        if getattr(existing, 'name', None) == name:
            print(f"Reusing existing agent: {name} ({getattr(existing, 'id', 'no-id')})")
            return {
                'name': name,
                'instructions': instructions,
                'model': model,
                'metadata': metadata,
                'details': existing,
                'version': None,
            }

    created = client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=model,
            instructions=instructions,
        ),
        description=f'{name} definition for pattern comparison notebook',
        metadata=metadata,
    )
    print(f"Created new agent version: {name} ({getattr(created, 'version', 'unknown-version')})")
    return {
        'name': name,
        'instructions': instructions,
        'model': model,
        'metadata': metadata,
        'details': None,
        'version': getattr(created, 'version', None),
    }


## 3. Agent definitions


In [15]:
AGENT_CONFIGS = {
    'sequential_fraud_analyst': {
        'name': 'PatternComparison-Sequential-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst. Analyze the reported transactions for fraud indicators, classify risk, identify suspicious patterns, and recommend investigation priorities.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_security_agent': {
        'name': 'PatternComparison-Sequential-SecurityAgent',
        'instructions': 'You are a banking account security specialist. Recommend immediate containment actions such as account holds, credential resets, card controls, fraud alerts, and monitoring steps ordered by urgency.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_credits_agent': {
        'name': 'PatternComparison-Sequential-CreditsAgent',
        'instructions': 'You are a provisional credit and claims specialist. Assess provisional credit eligibility, required documentation, verification checkpoints, and customer communication needs.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'sequential_resolution_reviewer': {
        'name': 'PatternComparison-Sequential-ResolutionReviewer',
        'instructions': 'You are a senior banking reviewer preparing the final resolution summary. Synthesize prior analysis into a clear action plan with investigation summary, protections, provisional credit view, open questions, and next steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'sequential'},
    },
    'handoff_router_agent': {
        'name': 'PatternComparison-Handoff-RouterAgent',
        'instructions': 'You are an orchestration router for banking fraud operations. Choose exactly one specialist: fraud_analyst, security_agent, or credits_agent. Return exactly two lines: SPECIALIST: <name> and REASON: <brief reason>.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_fraud_analyst': {
        'name': 'PatternComparison-Handoff-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst handling a routed case. Explain suspicious activity, transaction-level risk, likely fraud patterns, and what should be investigated first.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_security_agent': {
        'name': 'PatternComparison-Handoff-SecurityAgent',
        'instructions': 'You are a banking security specialist handling a routed case. Recommend urgent account protection actions, monitoring controls, and customer containment steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'handoff_credits_agent': {
        'name': 'PatternComparison-Handoff-CreditsAgent',
        'instructions': 'You are a provisional credit specialist handling a routed case. Evaluate provisional credit suitability, required evidence, timing considerations, and policy caveats.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'handoff'},
    },
    'concurrent_fraud_analyst': {
        'name': 'PatternComparison-Concurrent-FraudAnalyst',
        'instructions': 'You are a bank fraud analyst. Independently analyze the case and provide a concise fraud investigation perspective.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'concurrent_security_agent': {
        'name': 'PatternComparison-Concurrent-SecurityAgent',
        'instructions': 'You are a banking security specialist. Independently provide the account protection and containment perspective for the case.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'concurrent_credits_agent': {
        'name': 'PatternComparison-Concurrent-CreditsAgent',
        'instructions': 'You are a provisional credit specialist. Independently provide the claims and provisional credit perspective for the case.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'concurrent'},
    },
    'group_chat_fraud_analyst': {
        'name': 'PatternComparison-GroupChat-FraudAnalyst',
        'instructions': 'You are a fraud analyst participating in a group discussion. Build on previous messages, add new fraud insights, and avoid repeating earlier points.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'group_chat_security_agent': {
        'name': 'PatternComparison-GroupChat-SecurityAgent',
        'instructions': 'You are a banking security specialist participating in a group discussion. Build on previous messages with containment actions and operational safeguards.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'group_chat_credits_agent': {
        'name': 'PatternComparison-GroupChat-CreditsAgent',
        'instructions': 'You are a provisional credit specialist participating in a group discussion. Build on previous messages with claims, documentation, and provisional credit guidance.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
    'magentic_planner': {
        'name': 'PatternComparison-Magentic-Planner',
        'instructions': 'You are a strategic planner for banking fraud cases. Decompose the case into a prioritized task list. Return ONLY a JSON array of objects with keys: task_id, description, assigned_to (one of: researcher, writer, validator), priority (1-3). Example: [{"task_id": "T1", "description": "...", "assigned_to": "researcher", "priority": 1}]',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_researcher': {
        'name': 'PatternComparison-Magentic-Researcher',
        'instructions': 'You are a fraud case researcher. Given a specific subtask from the task ledger, investigate thoroughly and provide detailed findings. Be concise and evidence-focused. Reference the task_id you are working on.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_writer': {
        'name': 'PatternComparison-Magentic-Writer',
        'instructions': 'You are a fraud case report writer. Synthesize all research findings into a structured resolution document with sections: Executive Summary, Transaction Analysis, Risk Assessment, Recommended Actions, Provisional Credit Assessment, and Next Steps.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'magentic_validator': {
        'name': 'PatternComparison-Magentic-Validator',
        'instructions': 'You are a senior quality validator for banking fraud resolutions. Review the draft report for completeness, accuracy, and regulatory compliance. If acceptable, respond with DECISION: APPROVE followed by brief summary. If revisions needed, respond with DECISION: REVISE followed by specific feedback.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'magentic'},
    },
    'group_chat_moderator': {
        'name': 'PatternComparison-GroupChat-Moderator',
        'instructions': 'You are the moderator for a banking fraud resolution discussion. Review the conversation, identify gaps, and end with DECISION: continue or DECISION: conclude.',
        'metadata': {'source': 'pattern_comparison', 'pattern': 'group_chat'},
    },
}

for key, config in AGENT_CONFIGS.items():
    print(f"{key:32} -> {config['name']}")


sequential_fraud_analyst         -> PatternComparison-Sequential-FraudAnalyst
sequential_security_agent        -> PatternComparison-Sequential-SecurityAgent
sequential_credits_agent         -> PatternComparison-Sequential-CreditsAgent
sequential_resolution_reviewer   -> PatternComparison-Sequential-ResolutionReviewer
handoff_router_agent             -> PatternComparison-Handoff-RouterAgent
handoff_fraud_analyst            -> PatternComparison-Handoff-FraudAnalyst
handoff_security_agent           -> PatternComparison-Handoff-SecurityAgent
handoff_credits_agent            -> PatternComparison-Handoff-CreditsAgent
concurrent_fraud_analyst         -> PatternComparison-Concurrent-FraudAnalyst
concurrent_security_agent        -> PatternComparison-Concurrent-SecurityAgent
concurrent_credits_agent         -> PatternComparison-Concurrent-CreditsAgent
group_chat_fraud_analyst         -> PatternComparison-GroupChat-FraudAnalyst
group_chat_security_agent        -> PatternComparison-GroupChat-Secur

## 4. Create or find Foundry agents


In [16]:
agents = {}
for key, config in AGENT_CONFIGS.items():
    agents[key] = get_or_create_agent(
        client=client,
        name=config['name'],
        instructions=config['instructions'],
        model=MODEL_DEPLOYMENT,
        metadata=config['metadata'],
    )

print('\nResolved agents:')
for key, agent in agents.items():
    descriptor = getattr(agent.get('details'), 'id', None) or agent.get('version') or 'registered'
    print(f"- {key}: {agent['name']} ({descriptor})")


Reusing existing agent: PatternComparison-Sequential-FraudAnalyst (PatternComparison-Sequential-FraudAnalyst)
Reusing existing agent: PatternComparison-Sequential-SecurityAgent (PatternComparison-Sequential-SecurityAgent)
Reusing existing agent: PatternComparison-Sequential-CreditsAgent (PatternComparison-Sequential-CreditsAgent)
Reusing existing agent: PatternComparison-Sequential-ResolutionReviewer (PatternComparison-Sequential-ResolutionReviewer)
Reusing existing agent: PatternComparison-Handoff-RouterAgent (PatternComparison-Handoff-RouterAgent)
Reusing existing agent: PatternComparison-Handoff-FraudAnalyst (PatternComparison-Handoff-FraudAnalyst)
Reusing existing agent: PatternComparison-Handoff-SecurityAgent (PatternComparison-Handoff-SecurityAgent)
Reusing existing agent: PatternComparison-Handoff-CreditsAgent (PatternComparison-Handoff-CreditsAgent)
Reusing existing agent: PatternComparison-Concurrent-FraudAnalyst (PatternComparison-Concurrent-FraudAnalyst)
Reusing existing age

## 5. Helper -- Run an agent via chat completions and extract response text


In [17]:
def extract_response_text(response):
    content = response.choices[0].message.content
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                text = item.get('text')
            else:
                text = getattr(item, 'text', None)
            if text:
                parts.append(text)
        return '\n'.join(parts).strip()
    return str(content).strip()

def history_to_chat_messages(history_entries):
    chat_messages = []
    for entry in history_entries:
        speaker = entry['speaker']
        content = entry['content']
        role = 'user' if speaker in {'customer', 'user', 'task_ledger'} else 'assistant'
        message_content = content if speaker == 'customer' else f'[{speaker}] {content}'
        chat_messages.append({'role': role, 'content': message_content})
    return chat_messages

def run_agent(agent, task=None, messages=None, run_metadata=None):
    if messages is None:
        if task is None:
            raise ValueError('Provide either task or messages')
        messages = [{'role': 'user', 'content': task}]

    response = oai.chat.completions.create(
        model=agent.get('model', MODEL_DEPLOYMENT),
        messages=[
            {'role': 'system', 'content': agent['instructions']},
            *messages,
        ],
    )

    response_text = extract_response_text(response)
    return {
        'response_id': getattr(response, 'id', None),
        'model': getattr(response, 'model', None),
        'metadata': run_metadata or {},
        'response': response_text,
    }

def preview(text, limit=500):
    text = text or ''
    return text if len(text) <= limit else text[:limit] + '...'


## 6. Pattern 1 -- Sequential


In [7]:
start = time.perf_counter()
sequential_steps = []
sequential_history = [{'speaker': 'customer', 'content': BANKING_TASK}]

for step_name in [
    'sequential_fraud_analyst',
    'sequential_security_agent',
    'sequential_credits_agent',
    'sequential_resolution_reviewer',
]:
    result = run_agent(
        agent=agents[step_name],
        messages=history_to_chat_messages(sequential_history),
        run_metadata={'pattern': 'sequential', 'source': 'pattern_comparison'},
    )
    sequential_steps.append({
        'agent': step_name,
        'response': result['response'],
        'response_id': result['response_id'],
    })
    sequential_history.append({'speaker': step_name, 'content': result['response']})

results['Sequential'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'steps': sequential_steps,
    'final_output': sequential_steps[-1]['response'],
}

print(f"Sequential completed in {results['Sequential']['duration_seconds']}s")
for step in sequential_steps:
    print(f"\n[{step['agent']}]")
    print(preview(step['response']))


Sequential completed in 27.47s

[sequential_fraud_analyst]
To analyze the reported transactions for fraud indicators, we should break down each transaction individually and then assess them collectively for patterns and risk levels.

### Transaction Analysis

1. **$4,200 Wire Transfer to an Unknown Account**:
   - **Fraud Indicators**: Large, unexpected transfer to an unfamiliar account. Fraudulent wires often involve moving money quickly out of an account to a difficult-to-recover destination.
   - **Risk Level**: High. Wire transfers are generally hig...

[sequential_security_agent]
To address the unauthorized transactions reported by the customer, it's crucial to implement a series of immediate containment actions that will secure the account, investigate the potential fraud, and assess provisional credit. Below are the recommended steps ordered by urgency:

### Immediate Containment Actions

1. **Account Hold**:
   - **Objective**: Prevent further unauthorized access and transactio

## 7. Pattern 2 -- Handoff


In [18]:
start = time.perf_counter()
router_prompt = (
    f'{BANKING_TASK}\n\n'
    'Decide which specialist should take the first action. '
    'Return exactly two lines: SPECIALIST: <fraud_analyst|security_agent|credits_agent> and REASON: <reason>.'
)

router_result = run_agent(
    agent=agents['handoff_router_agent'],
    task=router_prompt,
    run_metadata={'pattern': 'handoff', 'source': 'pattern_comparison'},
)

router_text = router_result['response']
selected_specialist = 'handoff_fraud_analyst'
routing_map = {
    'fraud_analyst': 'handoff_fraud_analyst',
    'security_agent': 'handoff_security_agent',
    'credits_agent': 'handoff_credits_agent',
}
for token, mapped_agent in routing_map.items():
    if token in router_text.lower():
        selected_specialist = mapped_agent
        break

handoff_history = [
    {'speaker': 'customer', 'content': BANKING_TASK},
    {'speaker': 'router_agent', 'content': router_text},
]

specialist_result = run_agent(
    agent=agents[selected_specialist],
    messages=history_to_chat_messages(handoff_history),
    run_metadata={'pattern': 'handoff', 'source': 'pattern_comparison'},
)

results['Handoff'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'router_output': router_text,
    'selected_specialist': selected_specialist,
    'specialist_output': specialist_result['response'],
    'final_output': specialist_result['response'],
}

print(f"Handoff completed in {results['Handoff']['duration_seconds']}s")
print('\n[handoff_router_agent]')
print(router_text)
print(f"\n[selected specialist] {selected_specialist}")
print(preview(specialist_result['response']))


Handoff completed in 13.13s

[handoff_router_agent]
SPECIALIST: fraud_analyst  
REASON: To conduct a detailed investigation of the flagged transactions and determine the nature and extent of the fraud.

[selected specialist] handoff_fraud_analyst
As a bank fraud analyst, you should initiate a detailed investigation into the unauthorized transactions reported by the customer. Here is a breakdown of the situation, including suspicious activity, transaction-level risk, likely fraud patterns, and steps for immediate action:

### Suspicious Activity
1. **Wire Transfer to an Unknown Account:**
   - Amount: $4,200
   - Destination: Unknown account
   - Risk Factors: Wire transfers to unknown entities are high-risk due to their speed and irrever...


## 8. Pattern 3 -- Concurrent


In [9]:
start = time.perf_counter()
concurrent_agents = [
    'concurrent_fraud_analyst',
    'concurrent_security_agent',
    'concurrent_credits_agent',
]
concurrent_results = {}

def concurrent_worker(agent_key):
    return agent_key, run_agent(
        agent=agents[agent_key],
        task=BANKING_TASK,
        run_metadata={'pattern': 'concurrent', 'source': 'pattern_comparison'},
    )

with ThreadPoolExecutor(max_workers=len(concurrent_agents)) as executor:
    futures = [executor.submit(concurrent_worker, key) for key in concurrent_agents]
    for future in as_completed(futures):
        key, result = future.result()
        concurrent_results[key] = result

concurrent_summary = '\n\n'.join(
    f"[{key}]\n{concurrent_results[key]['response']}"
    for key in concurrent_agents
)

results['Concurrent'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'agent_outputs': {key: concurrent_results[key]['response'] for key in concurrent_agents},
    'final_output': concurrent_summary,
}

print(f"Concurrent completed in {results['Concurrent']['duration_seconds']}s")
for key in concurrent_agents:
    print(f"\n[{key}]")
    print(preview(concurrent_results[key]['response']))


Concurrent completed in 8.22s

[concurrent_fraud_analyst]
In analyzing this case from a fraud investigation perspective, several key considerations need to be addressed:

1. **Transaction Analysis**:
   - **$4,200 Wire Transfer**: Investigate the recipient account details, such as its history, location, and previous associations. This wire transfer to an unrecognized account suggests potential account takeover or internal fraud. Cross-reference recipient details with known fraud databases.
   - **$5,800 Online Purchase**: The nature of the purchase, th...

[concurrent_security_agent]
In addressing this case, it's critical to promptly secure the customer's account and initiate an investigation to prevent further unauthorized activity and assess potential reimbursements. Here’s a step-by-step account protection and containment action plan:

1. **Immediate Account Freeze and Verification:**
   - Freeze the affected business checking account immediately to prevent any further unauthorized 

## 9. Pattern 4 -- Group Chat


In [10]:
start = time.perf_counter()
group_sequence = [
    'group_chat_fraud_analyst',
    'group_chat_security_agent',
    'group_chat_credits_agent',
]
group_history = [{'speaker': 'customer', 'content': BANKING_TASK}]
group_rounds = []
max_rounds = 2

for round_number in range(1, max_rounds + 1):
    round_entries = []
    for agent_key in group_sequence:
        result = run_agent(
            agent=agents[agent_key],
            messages=history_to_chat_messages(group_history),
            run_metadata={'pattern': 'group_chat', 'source': 'pattern_comparison', 'round': round_number},
        )
        round_entries.append({'agent': agent_key, 'response': result['response']})
        group_history.append({'speaker': agent_key, 'content': result['response']})

    moderator_result = run_agent(
        agent=agents['group_chat_moderator'],
        messages=history_to_chat_messages(group_history),
        run_metadata={'pattern': 'group_chat', 'source': 'pattern_comparison', 'round': round_number},
    )
    round_entries.append({'agent': 'group_chat_moderator', 'response': moderator_result['response']})
    group_history.append({'speaker': 'group_chat_moderator', 'content': moderator_result['response']})
    group_rounds.append(round_entries)

    if 'decision: conclude' in moderator_result['response'].lower():
        break

results['Group Chat'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'rounds': group_rounds,
    'final_output': '\n\n'.join(
        f"[{entry['speaker']}] {entry['content']}" for entry in group_history
    ),
}

print(f"Group Chat completed in {results['Group Chat']['duration_seconds']}s")
for idx, round_entries in enumerate(group_rounds, start=1):
    print(f"\n=== Round {idx} ===")
    for entry in round_entries:
        print(f"\n[{entry['agent']}]")
        print(preview(entry['response'], limit=350))


Group Chat completed in 17.51s

=== Round 1 ===

[group_chat_fraud_analyst]
In such cases, it's crucial to initiate a comprehensive fraud investigation immediately to determine how the breach occurred. This should include reviewing the customer's recent login activity, IP addresses, and any irregular access times that could indicate unauthorized access.

For the specific transactions, each should be scrutinized for unique ...

[group_chat_security_agent]
[group_chat_information_security_specialist] Building on the comprehensive analysis, our immediate containment actions should focus on securing the account and any associated access points. Firstly, we should perform an immediate lock on the affected account, preventing any further transactions until identity verification and additional security me...

[group_chat_credits_agent]
[group_chat_provisional_credit_specialist] To address the customer's immediate financial concerns, we should proceed with evaluating the eligibility for provis

## 10. Pattern 5 -- Magentic (Task Ledger)

**How it works:** A Planner agent decomposes the case into a prioritized task ledger.
A Researcher works through each subtask. A Writer synthesizes findings into a report.
A Validator reviews and can request revisions -- creating an iterative feedback loop.

```
Customer Request
     |
Planner   ->  decomposes into subtask ledger (JSON)
     |
Researcher  ->  investigates each subtask
     |
Writer  ->  synthesizes resolution report
     |
Validator  ->  APPROVE or REVISE
     | (if REVISE, loops back to Writer, max 2 rounds)
Final Report
```

**Key insight:** Unlike Sequential (fixed pipeline, no feedback), Magentic uses a
**task ledger** for structured decomposition and a **validator loop** for iterative
quality refinement. The planner drives the strategy; the validator enforces quality.


In [11]:
import json as _json
import re

start = time.perf_counter()
magentic_steps = []
task_ledger = []

planner_result = run_agent(
    agent=agents['magentic_planner'],
    task=BANKING_TASK,
    run_metadata={'pattern': 'magentic', 'phase': 'planning'},
)
magentic_steps.append({'agent': 'magentic_planner', 'response': planner_result['response']})
print('[magentic_planner]')
print(preview(planner_result['response'], 400))

try:
    json_match = re.search(r'\[.*\]', planner_result['response'], re.DOTALL)
    task_ledger = _json.loads(json_match.group()) if json_match else []
except Exception:
    task_ledger = [
        {
            'task_id': 'T1',
            'description': 'Full case analysis',
            'assigned_to': 'researcher',
            'priority': 1,
        }
    ]
print(f'\nTask ledger: {len(task_ledger)} subtasks')

research_history = [{'speaker': 'customer', 'content': BANKING_TASK}]
research_history.append({'speaker': 'planner', 'content': planner_result['response']})

for task in sorted(task_ledger, key=lambda item: item.get('priority', 99)):
    subtask_prompt = f"SUBTASK {task.get('task_id', '?')}: {task.get('description', 'investigate')}"
    research_history.append({'speaker': 'task_ledger', 'content': subtask_prompt})

    r = run_agent(
        agent=agents['magentic_researcher'],
        messages=history_to_chat_messages(research_history),
        run_metadata={'pattern': 'magentic', 'phase': 'research', 'task_id': task.get('task_id', '?')},
    )
    research_history.append({'speaker': 'researcher', 'content': r['response']})
    magentic_steps.append({'agent': f"researcher ({task.get('task_id', '?')})", 'response': r['response']})
    print(f"\n[researcher - {task.get('task_id', '?')}] done ({len(r['response'])} chars)")

writer_history = research_history.copy()
writer_result = run_agent(
    agent=agents['magentic_writer'],
    messages=history_to_chat_messages(writer_history),
    run_metadata={'pattern': 'magentic', 'phase': 'writing'},
)
magentic_steps.append({'agent': 'magentic_writer', 'response': writer_result['response']})
print(f"\n[magentic_writer] done ({len(writer_result['response'])} chars)")

draft = writer_result['response']
max_revisions = 2
revision_count = 0

for rev_round in range(max_revisions + 1):
    val_history = research_history + [
        {'speaker': 'writer', 'content': draft},
    ]
    val_result = run_agent(
        agent=agents['magentic_validator'],
        messages=history_to_chat_messages(val_history),
        run_metadata={'pattern': 'magentic', 'phase': 'validation', 'round': rev_round},
    )
    magentic_steps.append({'agent': f'validator (round {rev_round})', 'response': val_result['response']})
    print(f"\n[validator round {rev_round}] {val_result['response'][:100]}")

    if 'APPROVE' in val_result['response'].upper():
        print('Validator approved!')
        break

    if rev_round < max_revisions:
        revision_count += 1
        revise_history = research_history + [
            {'speaker': 'writer', 'content': draft},
            {'speaker': 'validator', 'content': val_result['response']},
        ]
        revised = run_agent(
            agent=agents['magentic_writer'],
            messages=history_to_chat_messages(revise_history),
            run_metadata={'pattern': 'magentic', 'phase': 'revision', 'round': rev_round},
        )
        draft = revised['response']
        magentic_steps.append({'agent': f'writer (revision {revision_count})', 'response': draft})
        print(f"[writer revision {revision_count}] done ({len(draft)} chars)")

results['Magentic'] = {
    'duration_seconds': round(time.perf_counter() - start, 2),
    'steps': magentic_steps,
    'task_ledger': task_ledger,
    'revision_count': revision_count,
    'final_output': draft,
}

print(f"\nMagentic completed in {results['Magentic']['duration_seconds']}s")
print(f"Task ledger: {len(task_ledger)} subtasks | Revisions: {revision_count}")
print('\n--- Final Report Preview ---')
print(preview(draft, 600))


[magentic_planner]
```json
[
    {"task_id": "T1", "description": "Gather details of reported unauthorized transactions from the customer’s account statement and verify transactions", "assigned_to": "researcher", "priority": 1},
    {"task_id": "T2", "description": "Contact the wire transfer destination bank to confirm recipient details and link with possible fraud", "assigned_to": "researcher", "priority": 1},
    ...

Task ledger: 9 subtasks

[researcher - T1] done (2373 chars)

[researcher - T2] done (2460 chars)

[researcher - T3] done (2570 chars)

[researcher - T5] done (2778 chars)

[researcher - T4] done (2490 chars)

[researcher - T6] done (3638 chars)

[researcher - T7] done (2180 chars)

[researcher - T8] done (2964 chars)

[researcher - T9] done (2742 chars)

[magentic_writer] done (2896 chars)

[validator round 0] DECISION: APPROVE

The report provides a comprehensive review of the unauthorized transactions, deta
Validator approved!

Magentic completed in 79.51s
Task ledge

## 11. Comparison

Timing and summary table for all five orchestration patterns.


In [12]:
comparison = [
    ('Sequential', 'Agents run in series, each builds on previous', 'Multi-step workflows with dependencies'),
    ('Handoff', 'Router picks one specialist', 'Routing to domain experts, call centers'),
    ('Concurrent', 'All agents run in parallel', 'Independent analyses needed fast'),
    ('Group Chat', 'Agents discuss in rounds', 'Complex decisions requiring debate'),
    ('Magentic', 'Planner, task ledger, writer, validator loop', 'Complex cases needing structured decomposition and quality gates'),
]

def summary_count(pattern_name, payload):
    if pattern_name == 'Sequential':
        return len(payload.get('steps', []))
    if pattern_name == 'Handoff':
        return 2 if payload else 0
    if pattern_name == 'Concurrent':
        return len(payload.get('agent_outputs', {}))
    if pattern_name == 'Group Chat':
        return sum(len(round_entries) for round_entries in payload.get('rounds', []))
    if pattern_name == 'Magentic':
        return len(payload.get('steps', []))
    return 0

print(f"{'Pattern':<12} {'Time (s)':>10} {'Steps':>8}  {'How It Works'}")
print(f"{'-' * 12} {'-' * 10} {'-' * 8}  {'-' * 55}")
for name, how, best_for in comparison:
    payload = results.get(name)
    if payload:
        duration = payload.get('duration_seconds', '--')
        steps = summary_count(name, payload)
        print(f"{name:<12} {duration:>10} {steps:>8}  {how}")
    else:
        print(f"{name:<12} {'--':>10} {'--':>8}  {how}")

print('\nBest fit by pattern:')
for name, how, best_for in comparison:
    print(f'- {name}: {best_for}')


Pattern        Time (s)    Steps  How It Works
------------ ---------- --------  -------------------------------------------------------
Sequential        27.47        4  Agents run in series, each builds on previous
Handoff            8.98        2  Router picks one specialist
Concurrent         8.22        3  All agents run in parallel
Group Chat        17.51        4  Agents discuss in rounds
Magentic          79.51       12  Planner, task ledger, writer, validator loop

Best fit by pattern:
- Sequential: Multi-step workflows with dependencies
- Handoff: Routing to domain experts, call centers
- Concurrent: Independent analyses needed fast
- Group Chat: Complex decisions requiring debate
- Magentic: Complex cases needing structured decomposition and quality gates


## 12. Cleanup

Delete the persistent Foundry agents created for this notebook when you no longer need them.
Set `DELETE_PATTERN_COMPARISON_AGENTS=1` in `.env` to actually remove them.


In [ ]:
delete_agents = os.getenv('DELETE_PATTERN_COMPARISON_AGENTS', '0') == '1'

if not delete_agents:
    print('Skipping cleanup. Set DELETE_PATTERN_COMPARISON_AGENTS=1 in .env to delete the created agents.')
    print('Tracked agents:')
    for key, agent in agents.items():
        print(f"- {key}: {agent['name']}")
else:
    deleted = set()
    for key, agent in agents.items():
        if agent['name'] in deleted:
            continue
        client.agents.delete(agent['name'])
        deleted.add(agent['name'])
        print(f"Deleted {key}: {agent['name']}")
